# Spark SQL Learning 
##  By knowing the process of Read and write, become a  Data INGESTION / Data Ingress Developer 
##  By knowing the process of exporting the data, become a  Data Egress Developer  

- Exporting data from databases or applications
- Building APIs that send data to external systems
- Migrating data between cloud platforms
- Managing data transfer pipelines

connecting various sources (files or filesystem , db , dwh , api ,.. ) and loading data into storage env(data lake)

- csv
- json
- xml
- parquet 
- ORC
- sql
etc....

In [0]:
spark.version

## Few Facts about Unity Catalog

Unity Catalog is centralized governance solution for managing data, tables, files, machine learning models, permissions, and auditing across all workspaces.


In Unity Catalog, data objects are referenced as Three-Level Namespace


Catalog -> per domain or environemnt 

schema -> database 

tables , views , functions , volume 

Volume -  Non tabular data (files) , goverened access 

Catalog >> Schema  >> 
                    Table
                    View
                    functions
                    volume 

Volumes are used for managing Non tabular data (files)

In [0]:
%sql
create catalog if not exists izwd37dev;

create schema if not exists izwd37dev.wd37db;

create volume if not exists izwd37dev.wd37db.rawdatta;

DBFS - Databricks File system
distribuited virtual file system , linux posix format runninng on top of your cloud storages

- /Volumes/catalog/schema/volume-name/path_to_file
- dbfs:/ - uri -> uniform resource identifier (dbricks file system )
- hdfs:/ -> hadoop distruibuited file system file:/ -> local file
- s3a:/ -> aws s3
- gcs:/ -> google storage
- adls:/ -> azure datalake

In [0]:
%fs ls /Volumes/izwd37dev/wd37db/rawdatta/

In [0]:
dbutils.fs.ls("dbfs:/Volumes/izwd37dev/wd37db/rawdatta")


In [0]:
%fs head "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp.csv"

In [0]:
%fs mkdirs "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv"

In [0]:
dbutils.fs.mkdirs("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/sales")

Read data from Spark

Spark session

from pyspark.sql import SparkSession

spark=SparkSession.builder.getOrCreate()

In [0]:
print(spark)

create Dataframe from storage (files / dir ) To read the delimited data from any storage (dbfs , hdfs , lfs , cloud storages )

spark.read.csv option -> create a dataframe

-- csv is the built in source

**built in Source**
- CSV
- JSON
- XML
- parquet
- delta
- ORC
- Excel
- jdbc
- table

## Defaults
spark.read.csv

default options :

- header = False

- default cols = _c0 , _c1 _c2

- delimiter = ","

- default data type for all columns = string

In [0]:

empdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv") # file path
print(type(empdata)) # dataframe
empdata.show() # similar to collect -> action )
# show action , display default 20 records 
     

In [0]:
empdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv",header=True) # file path
print(type(empdata)) # dataframe
empdata.show() 
empdata.printSchema()

In [0]:
empdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp.csv",header=True,inferSchema=True) # file path
print(type(empdata)) # dataframe
# view few or more records 
empdata.show(5,True)
empdata.printSchema()

In [0]:
display(empdata)

In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv") # file path
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()


In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv").toDF("Br_id","Emp_ID","FName","LName","Hire_Date","Profession")
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()

In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv",inferSchema=True,header=True).toDF("Br_id","Emp_ID","FName","LName","Hire_Date","Profession")
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()
empdata_df.count()  # here count is 7 because first row considered as header

In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv",inferSchema=True).toDF("Br_id","Emp_ID","FName","LName","Hire_Date","Profession")
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()
empdata_df.count()     # header =False so, first row not considered as Header so count is 8


**Different Delimiter**

default is ","

In [0]:
empdata_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter.csv",sep="|",inferSchema=True).toDF("Br_id","Emp_ID","FName","LName","Hire_Date","Profession")
print(type(empdata_df)) # dataframe
empdata_df.show(5)
empdata_df.printSchema()
empdata_df.count()

spark.read.csv()

-> required input path

-> path could be a file or dir , list of dir ...

In [0]:
empdata_df=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv",header=True)
empdata_df.show(5)

In [0]:
empdata_df=spark.read.csv(path=["/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_without_header.csv","/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv","/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter.csv"],header=True)
empdata_df.show(10)
empdata_df.count()

In [0]:
%fs head "/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter.csv"

**Create sub directory using mkdirs**

In [0]:
%fs mkdirs "/Volumes/izwd37dev/wd37db/rawdatta/sales/Sales_Chennai"

In [0]:
%fs mkdirs "/Volumes/izwd37dev/wd37db/rawdatta/sales/Sales_Delhi"

In [0]:
sales_data_df=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/",header=True,inferSchema=True,recursiveFileLookup=True)

sales_data_df.show(100)
sales_data_df.printSchema()
sales_data_df.count()  # 1022



In [0]:
sales_data_df=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/sales",header=True,inferSchema=True,recursiveFileLookup=True,pathGlobFilter="Sales*")

sales_data_df.show(100)
sales_data_df.printSchema()
sales_data_df.count()   #count :1000




# inferSchema -> generating schema by reading the entire data 
- to avoid reading the data for genrating schema , - performance issue 
- when we are going with inferschema its more dynamic  - data quality issue 


Creating schema in 2 ways.

- using ddl format (available from spark 3x version)
- using programming format (typically using for long days)

In [0]:
creating_schema="empid integer,firstname string,lastname string,profession string,age integer,location string"
sales_data_df=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/sales/Sales_Chennai",header=True,schema=creating_schema)
sales_data_df.show(10)
sales_data_df.printSchema()

In [0]:
creating_schema="branch_id integer,emp_id integer,firstname string,Joining_Date date,lastname string,profession string"
sales_data_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter_ageupdated.csv",schema=creating_schema,sep="|")
sales_data_df.show(5)
sales_data_df.printSchema()


**create schema using programming method**

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
schema_create=StructType([
    StructField("branch_id",IntegerType()),
    StructField("emp_id",IntegerType()),
    StructField("firstname",StringType()),
    StructField("joining_date",StringType()),
    StructField("lastname",StringType()),
    StructField("profession",StringType())
])

sales_data_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter_ageupdated.csv",schema=schema_create,sep="|")
sales_data_df.show(5)
sales_data_df.printSchema()      #joining_date is stringtype but we want to convert it into date type so using next approach


**Create schema using programming**
- StructType for row

- StructField for columns

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import to_date
schema_create=StructType([
    StructField("branch_id",IntegerType()),
    StructField("emp_id",IntegerType()),
    StructField("firstname",StringType()),
    StructField("joining_date",StringType()),
    StructField("lastname",StringType()),
    StructField("profession",StringType())
])


sales_data_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter_ageupdated.csv",schema=schema_create,sep="|")
sales_data_df = sales_data_df.withColumn(
    "joining_date",
    to_date("joining_date", "dd/MM/yyyy")
)
sales_data_df.show(5)
sales_data_df.printSchema()

# **_Read data from JSON_**

In [0]:
%fs head "/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json"


In [0]:

emp_scehma="branch_id integer,emp_id integer,firstname string,lastname string,profession string,age integer, joining_date string,location string"
spark_json_df=spark.read .option("multiline", "true").option("delimiter",",").json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",schema=emp_scehma)
spark_json_df.show(5)
spark_json_df.printSchema()


# connect external source -  genric way to read data using spark

- spark.read.option("k","v").format("source").load()
- csv 
- df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp.csv")
 


In [0]:
cust_df=spark.read.option("header","True").option("inferSchema","True").option("delimiter",",").format("csv").load("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv")

cust_df.show()

In [0]:
emp_json_df=spark.read.schema(emp_scehma).format("json").load("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json")

emp_json_df.show()
emp_json_df.printSchema()
     

**create dataframe from any delimited file, comes from any storage**
- local
- hdfs
- dbfs (databricks file system)
- cloud storages (s3,gcs,adls)

df=spark.read.csv()

or

df=spark.read.format("csv").load("url")

inline option --> option("inferschema","True") - passes string value, we can give both "True","TRUE","true"

header =True - passes boolean value (always capital T for True)

In [0]:
df=spark.read.option("inferschema","True").csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp.csv",header=True)
df.show()
df.printSchema()

### built in source: csv ->delimited file coming from any storages(local,hdfs,cloud object storages)

### semi structured data -> xml /json --spark.read.xml or spark.read.json

### structured/ databricks sql/ lakehouse table ->spark.read.table or spark.sql("sql queries")

### From rdbc ->spark.read.jbdc (study as part of cloud)

reading comma seperated data (eid,ename,address)

address column also have comma seperated

source sending header/ footer


if address column have comma seperated with street,city then use quote for address

quote='"'  or "'"

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv


In [0]:
emp_schema="eid integer,ename string,address string"


emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv",sep=",",header=True,schema=emp_schema,quote="'",escape="|")
emp_df.show(10,False)
emp_df.printSchema()

**if address also have single quote in its data, then use escape="|'s"**

In [0]:


emp_schema="eid integer,ename string,address string"


emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp3.csv",sep="|",header=True,schema=emp_schema,quote="'")
emp_df.show(10)
emp_df.show(10,False)  #show default first 20 records, here false is mentioned not to truncate the column values, truncate =False
emp_df.printSchema()

**if data file have header and footer then use  **

mark the header and footer using #

**escape="#"**

In [0]:
emp_schema="eid integer,ename string,address string"


emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv",sep=",",header=True,schema=emp_schema,quote="'",escape="|")
emp_df.show(10,False)
emp_df.printSchema()

adding header and footer in data file, how to read in those cases. use comment ="#"

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv

In [0]:
emp_schema="eid integer,ename string,address string"


emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv",sep=",",header=True,schema=emp_schema,quote="'",escape="|",comment="#")
emp_df.show(10,False)
emp_df.printSchema()

write the data to new delimited csv file 
now emp_df have , in address column
save emp_df in emp_write

In [0]:
emp_df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/csv_write/emp5.csv",sep=",")

In [0]:
emp_df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/csv_write/emp5.csv",sep=",",quote="'",escape="~")


**Create dataframe from python objects using "createDataFrame"**

programmitically create dataframe


In [0]:
data=[(1,"anu,shiva",23),(2,"ram,kumar",24),(3,"sam,jar",25),(4,"raj,kumar",26),(5,"jai,kumar",27)]
col_name=["id","Name","age"]
df=spark.createDataFrame(data,col_name)
df.show()
df.printSchema()


In [0]:
df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/python_df_out/df.csv",sep=",",quote="'")


create json dataframe with multiline json data and date

**mention dateformat =dd/mm/yyyy**

**multiline=True**

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/json/emp.json


In [0]:
json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",multiLine=True)
json_df.show(5,False)
json_df.printSchema()


**Create JSON schema using programming method**

spark default date format - "yyyy-mm-dd"

it will always show date format like "yyyy-mm-dd"

In [0]:
import json
from pyspark.sql.types import *
from pyspark.sql.functions import *

json_schema=StructType([
    StructField("age",LongType()),
    StructField("branch_id",LongType(),True),
    StructField("emp_id",LongType(),True),
    StructField("firstname",StringType(),True),
    StructField("joining_date",DateType(),True),
    StructField("lastname",StringType(),True),
    StructField("location",StringType(),True),
    StructField("profession",StringType(),True)
])
json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",multiLine=True,schema=json_schema,dateFormat="dd/mm/yyyy")
json_df.show(5,False)
json_df.printSchema()

**Create JSON schema using DDL method**

In [0]:
json_emp_schema="age int,branch_id int,emp_id int,firstname string,joining_date date,lastname string,location string,profession string"
json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",multiLine=True,schema=json_emp_schema,dateFormat="dd/mm/yyyy")
json_df.show(5,False)
json_df.printSchema()

In [0]:
json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",multiLine=True,schema=json_schema,dateFormat="dd/mm/yyyy")
json_df.write.mode("overwrite").saveAsTable("izwd37dev.wd37db.json_emp")

In [0]:
emp_csv_df=spark.read.format("csv").option("sep",",").option("header","True").option("comment","#").option("escape","|").load("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv")
emp_csv_df.show(5,False)

In [0]:
emp_csv_df=spark.read.format("csv").option("sep",",").option("header","True").option("comment","#").option("escape","|").option("quote","'").load("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv")
emp_csv_df.show(5,False)

**different modes**
- Permissive - permit the records and marked as null use "columnNameOfCorruptRecord" and create new column for this in schema
- failFast - fail the application
- dropMalFormed - drop the failed records

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/student.csv

In [0]:
student_schema="empid integer,firstname string,lastname string,profession string,age integer,location string"
student_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/student.csv",header=True,schema=student_schema)
student_df.count()
student_df.show()
student_df.printSchema()


In [0]:
student_schema="empid integer,firstname string,lastname string,profession string,age integer,location string"
student_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/student.csv",header=True,schema=student_schema,mode="PERMISSIVE")
student_df.count()
student_df.show()
student_df.printSchema()

In [0]:
student_schema="empid integer,firstname string,lastname string,profession string,age integer, location string"
student_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/student.csv",header=True,schema=student_schema,mode="failFast")
student_df.count()
student_df.show()
student_df.printSchema()

In [0]:
student_schema="empid integer,firstname string,lastname string,profession string,age integer,location string"
student_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/student.csv",header=True,schema=student_schema,mode="dropMalformed")
student_df.count()
student_df.show()
student_df.printSchema()

In [0]:
student_schema="empid integer,firstname string,lastname string,profession string,age integer,location string,error_rec string"
student_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/student.csv",header=True,schema=student_schema,mode="PERMISSIVE",columnNameOfCorruptRecord="error_rec")
student_df.count()
student_df.show()
student_df.printSchema()

option -> mode
**if bad record (not matching the schema )**
- allow - default (**permissive** -> permitting , RCA later)
- fail (**failfast** - immediately fail the whole job)
- ignore (**dropmalformed** - blindly drop that record proceed)





In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter_ageupdated.csv

In [0]:
schema_struct_csv="id integer,year integer,name string,joining_date date,lname string,position string,error_rec string"
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp_differnt_Delimiter_ageupdated.csv",sep="|",schema=schema_struct_csv,dateFormat="dd/mm/yyyy",mode="PERMISSIVE",columnNameOfCorruptRecord="error_rec")
df.show()
df.printSchema()
df.count()

- Built in Source - csv -> delimited file coming from any storages (local , hdfs , cloud object storage (s3))
- semi structure format - json / xml ->
- sturucture / databricks sql / lakehouse table -> spark.read.table() / spark.sql("sql query ")
- Rdbms Source -> spark.read.jdbc() -( cover as part of cloud )

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv

In [0]:

import pyspark.sql.functions as F
from pyspark.sql.types import *
schema=StructType([
    StructField("eid",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("addresss",StringType(),True)])
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv",schema=schema)
df.show()
df.printSchema()
df.count()


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
schema=StructType([
    StructField("eid",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("addresss",StringType(),True)])
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/emp2.csv",header=True,schema=schema,comment="#",escape="|",quote="'")
df.show()
df.printSchema()
df.count()

Create python objects programmatically



In [0]:
data=[(100,"anu",26),(101,"shiva",30),(102,"rushika",6)]
column=["eid","name","age"]
df=spark.createDataFrame(data,column)
df.show()
df.printSchema()
df.count()
df.write.mode("overwrite").json("/Volumes/izwd37dev/wd37db/rawdatta/json_write_fromPython/")

In [0]:
%fs ls /Volumes/izwd37dev/wd37db/rawdatta/json_write_fromPython/

In [0]:
df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/json_write_fromPython/")
df.show()


- # Json options explore
- # yyyy-MM-dd - spark date format 
- # allow single quotes
- # allow unquotedField names

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/json/emp1.json

In [0]:
schema_json="branch_id integer, emp_id integer, firstname string, lastname string, profession string, age integer, joining_date date, location string"
df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp1.json",schema=schema_json,dateFormat="mm/dd/yyyy",multiLine=True,allowSingleQuotes=True,allowUnquotedFieldNames=True)
df.show()
df.printSchema()
df.count()

## Schema Evolution -  changes in the scehma


In [0]:
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo/",header=True)
df.show()
df.count()

In [0]:
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo/day1.csv",header=True)
df.count()
df.show()
df.write.mode("overwrite").parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_csv_out")
df=spark.read.parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_csv_out")
df.show()
df.count()


In [0]:
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo/day2.csv",header=True,quote="'",escape="|")
df.count()
df.show()
df.write.mode("append").parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_csv_out")
df.count()
df.show()

In [0]:
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo/day3.csv",header=True)
df.count()
df.show()
df.write.mode("append").parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_csv_out")
df.count()
df.show()

**use mergeSchema in parquet to merge all  file schemas**

In [0]:

df=spark.read.parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_csv_out",mergeSchema=True)
df.count()
df.show(10,False)